# Randomized Cross-Validation with a Pipeline

## Purpose

Build a scikit-learn pipeline that preprocesses Titanic data and tune its Random Forest with `RandomizedSearchCV`. The notebook shows the concrete steps, parameters, and evaluation calls for a leakage-safe workflow.

## Contents

1. [Load the data](#load-the-data)
2. [Split the data](#split-the-data)
3. [Define preprocessing and the pipeline](#define-preprocessing-and-the-pipeline)
4. [Choose the Random Forest search space](#choose-the-random-forest-search-space)
5. [Run randomized cross-validation](#run-randomized-cross-validation)
6. [Evaluate once on the test set](#evaluate-once-on-the-test-set)

In [ ]:
import pandas as pd
from scipy.stats import randint

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## Load the data

Load the Titanic data from OpenML and select four predictors. Before preprocessing, `age` and `fare` are numeric and contain missing values; `sex` and `embarked` are categorical, with some missing `embarked` values. The target is `survived`.

In [ ]:
titanic = fetch_openml("titanic", version=1, as_frame=True).frame

numeric_features = ["age", "fare"]
categorical_features = ["sex", "embarked"]
feature_names = numeric_features + categorical_features

X = titanic[feature_names]
y = titanic["survived"]

X.head()

In [ ]:
X.isna().sum()

## Split the data

Set aside the test data before tuning. The search below uses only the training split; use the test split only for the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0,
)

X_train.shape, X_test.shape

## Define preprocessing and the pipeline

After preprocessing, every feature is numeric and missing values are filled: numeric columns use their training-fold medians, while categorical columns use their training-fold most-frequent values and are then one-hot encoded. The imputer statistics and the encoder's category vocabulary are estimated from data during `fit`; one-hot encoding expands categories into indicator columns. These estimates must be learned inside each cross-validation training fold, not once before cross-validation. Scaling is not required for a Random Forest.

In [ ]:
numeric_preprocessing = SimpleImputer(strategy="median")
categorical_preprocessing = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_features),
        ("categorical", categorical_preprocessing, categorical_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(random_state=0)),
    ]
)

pipeline

## Choose the Random Forest search space

Tune settings that control the forest: 
- `n_estimators` is the number of trees; 
- `max_depth`, `min_samples_split`, and `min_samples_leaf` constrain tree growth; 
- `max_features` controls the number of candidate features considered at each split; and 
- `bootstrap` selects whether each tree is trained from a bootstrap sample. 
These are hyperparameters chosen before fitting. The fitted split points and tree structures are learned from the training data. 

In a pipeline, prefix each parameter with its step name and two underscores, such as `model__max_depth`.

In [ ]:
param_distributions = {
    "model__n_estimators": randint(100, 501),
    "model__max_depth": [None, *range(3, 16)],
    "model__min_samples_split": randint(2, 21),
    "model__min_samples_leaf": randint(1, 11),
    "model__max_features": ["sqrt", "log2", None],
    "model__bootstrap": [True, False],
}

## Run randomized cross-validation

`n_iter` sets the number of sampled configurations. Each is scored with five stratified folds using macro-F1; `refit=True` refits the best pipeline on all training data. Each fold fits its own preprocessing before fitting its forest.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    refit=True,
    n_jobs=-1,
    random_state=0,
)

search.fit(X_train, y_train)

In [ ]:
print(f"Best mean CV macro-F1: {search.best_score_:.3f}")
pd.Series(search.best_params_, name="selected value")

In [ ]:
results = pd.DataFrame(search.cv_results_)
result_columns = [
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_split",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "param_model__bootstrap",
    "mean_test_score",
]

results.sort_values("mean_test_score", ascending=False)[result_columns].head().round(3)

## Evaluate once on the test set

`search.best_estimator_` is the selected pipeline refitted on all training data. Pass it the raw test features; the fitted preprocessing steps transform them before prediction.

In [ ]:
y_pred = search.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Test macro-F1: {f1_score(y_test, y_pred, average='macro'):.3f}")

## Workflow checks

- Keep every data-fitted transformation inside the pipeline passed to cross-validation.
- Tune with the training split; evaluate on the held-out test split once.